In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
df4 = pd.read_pickle('../data/df4_with_regions.pkl')
print(f"Loaded df4_with_regions: {df4.shape}")

for visualisation suite) 
- function to call values of current transfer timing distribution by age group and time of day
- function to call distribution of spatial transfer location pairs
- temporal transfer pattern distribution (y axis average transfer time, x axis hour of day) 


In [ ]:
# Cell 3: Verify filter value
print(df4['final_termination_reason_spatial'].value_counts().head(10))

In [ ]:
# Cell 4: Filter to confirmed transfers only (reusable base)
df_transfers = df4[
    df4['final_termination_reason_spatial'] == 'candidate_transfer'
].copy()

df_transfers['age_group'] = df_transfers['PATRON_CATG_DESC_TXT'].map({
    'Student':        '7-19',
    'Adult':          '20-59',
    'Senior Citizen': '60+'
})

df_transfers['hour_of_day'] = df_transfers['next_ENTRY_TM'].dt.hour

print(f"Confirmed transfers: {len(df_transfers):,}")
print(f"Null age_group: {df_transfers['age_group'].isna().sum():,}")
print(f"Null hour_of_day: {df_transfers['hour_of_day'].isna().sum():,}")
print(f"Null time_gap_mins: {df_transfers['time_gap_mins'].isna().sum():,}")

In [ ]:
# Cell 5: Function 1 — Transfer timing distribution by age group and hour of day
# Output: avg transfer time (time_gap_mins) per age group × hour
# CSV columns: age_group, hour_of_day, avg_transfer_time_mins

def trf_time_distribution_csv(df_trf):
    result = (
        df_trf.groupby(['age_group', 'hour_of_day'])['time_gap_mins']
        .mean()
        .reset_index(name='avg_transfer_time_mins')
        .sort_values(['age_group', 'hour_of_day'])
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_time_distribution.csv', index=False)
    print(f"Saved trf_time_distribution.csv: {result.shape}")
    return result

def trf_time_distribution(file='../data/trf_time_distribution.csv'):
    return pd.read_csv(file)

In [ ]:
# Cell 6: Run Function 1
trf_time_dist_df = trf_time_distribution_csv(df_transfers)
print(trf_time_dist_df)

In [ ]:
# Cell 7: Function 2 — Spatial transfer location pairs (actual transfer links)
# Output: top N most common transfer links from current alighting point to next boarding point
# CSV columns: transfer_from, transfer_to, count

def trf_pair_distribution_csv(df_trf, top_n=50):
    result = (
        df_trf[
            df_trf['DEST_STATION_NAME'].notna() &
            df_trf['next_orig_station'].notna()
        ]
        .groupby(['DEST_STATION_NAME', 'next_orig_station'])
        .size()
        .reset_index(name='count')
        .rename(columns={
            'DEST_STATION_NAME': 'transfer_from',
            'next_orig_station': 'transfer_to'
        })
        .sort_values('count', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    result.to_csv('../data/trf_pair_dist.csv', index=False)
    print(f"Saved trf_pair_dist.csv: {result.shape}")
    return result


def trf_pair_distribution(file='../data/trf_pair_dist.csv'):
    return pd.read_csv(file)

In [ ]:
'''# Cell 7: Function 2 — Spatial transfer location pairs (top-N, long format)
# Output: top N most common origin-destination transfer pairs with counts
# CSV columns: ORIG_STATION_NAME, DEST_STATION_NAME, count

def trf_pair_distribution_csv(df_trf, top_n=50):
    result = (
        df_trf.groupby(['ORIG_STATION_NAME', 'DEST_STATION_NAME'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_pair_dist.csv', index=False)
    print(f"Saved trf_pair_dist.csv: {result.shape}")
    return result

def trf_pair_distribution(file='../data/trf_pair_dist.csv'):
    return pd.read_csv(file)'''

In [ ]:
# Cell 8: Run Function 2
trf_pair_dist_df = trf_pair_distribution_csv(df_transfers)
print(trf_pair_dist_df.head(10))

In [ ]:
# Cell 9: Function 3 — Temporal transfer pattern by hour of day, broken down by patron
# Output: avg transfer time per hour × patron category
# CSV columns: hour_of_day, PATRON_CATG_DESC_TXT, avg_transfer_time_mins

def trf_pattern_distribution_csv(df_trf):
    result = (
        df_trf.groupby(['hour_of_day', 'PATRON_CATG_DESC_TXT'])['time_gap_mins']
        .mean()
        .reset_index(name='avg_transfer_time_mins')
        .sort_values(['hour_of_day', 'PATRON_CATG_DESC_TXT'])
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_pattern_distribution.csv', index=False)
    print(f"Saved trf_pattern_distribution.csv: {result.shape}")
    return result

def trf_pattern_distribution(file='../data/trf_pattern_distribution.csv'):
    return pd.read_csv(file)

In [ ]:
# Cell 10: Run Function 3
trf_pattern_dist_df = trf_pattern_distribution_csv(df_transfers)
print(trf_pattern_dist_df)

In [ ]:
# Frontend query functions (read from pre-computed CSVs)

_data_dir = '../data'

def get_trf_time_distribution(age_group=None, hour=None):
    """
    Returns avg transfer time distribution by age group and hour of day.

    Inputs:
    - age_group: None (all), or one of '7-19', '20-59', '60+'
    - hour:      None (all), or int 0-23

    Returns: DataFrame with [age_group, hour_of_day, avg_transfer_time_mins]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_time_distribution.csv'))

    if age_group is not None:
        df = df[df['age_group'] == age_group]
    if hour is not None:
        df = df[df['hour_of_day'] == hour]

    return df.reset_index(drop=True)


'''def get_trf_pair_distribution(orig_station=None, dest_station=None):
    """
    Returns top transfer station pairs by count.

    Inputs:
    - orig_station: None (all), or filter by origin station name string
    - dest_station: None (all), or filter by destination station name string

    Returns: DataFrame with [ORIG_STATION_NAME, DEST_STATION_NAME, count]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_pair_dist.csv'))

    if orig_station is not None:
        df = df[df['ORIG_STATION_NAME'] == orig_station]
    if dest_station is not None:
        df = df[df['DEST_STATION_NAME'] == dest_station]

    return df.reset_index(drop=True)'''

def get_trf_pair_distribution(transfer_from=None, transfer_to=None):
    """
    Returns top transfer location pairs by count.

    Inputs:
    - transfer_from: None (all), or filter by current alighting location
    - transfer_to:   None (all), or filter by next boarding location

    Returns: DataFrame with [transfer_from, transfer_to, count]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_pair_dist.csv'))

    if transfer_from is not None:
        df = df[df['transfer_from'] == transfer_from]
    if transfer_to is not None:
        df = df[df['transfer_to'] == transfer_to]

    return df.reset_index(drop=True)


def get_trf_temporal_pattern(patron=None):
    """
    Returns avg transfer time by hour of day for the temporal pattern chart.

    Inputs:
    - patron: None (all), or one of 'Adult', 'Student', 'Senior Citizen'

    Returns: DataFrame with [hour_of_day, PATRON_CATG_DESC_TXT, avg_transfer_time_mins]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_pattern_distribution.csv'))

    if patron is not None:
        df = df[df['PATRON_CATG_DESC_TXT'] == patron]

    return df.reset_index(drop=True)